# Trích xuất shot và caption trên Kaggle
Bật **Internet** và **GPU**. Notebook clone nhánh `vbs-dev`, chọn dải ID V3C, tải tối đa 10 video `<video_id>.mp4` mỗi lô từ cùng folder Drive của ASR. Pipeline tạo `scenes/V3C/<video_id>.scenes.txt` (shot general, frame đầu/cuối inclusive) và `captions_2b/<run_id>/V3C/<video_id>.json` (chỉ `video_id`, `start`, `end`, `caption`). Notebook upload **chỉ JSON caption** lên folder Drive đích, một lệnh rclone mỗi lô; file shot và JSON cũng được giữ trong workspace Kaggle của phiên chạy.

OmniShotCut được giải phóng trước khi nạp model caption đã chọn. File `.partial.json` lưu tiến độ từng caption, nên chạy lại cell cuối tiếp tục khi lỗi. Không tạo môi trường ảo; thư viện và model cache được dùng lại trong cùng phiên Kaggle. Cell prompt ghi đè cấu hình trong bản clone để thử VideoLLaMA3-2B mà không cần push prompt lên GitHub.

Sau shot detection, SpeedPredictor chạy NVDEC + resize + inference trên GPU, gắn nhãn trong RAM cho toàn bộ lô. Giải phóng predictor trước khi nạp model caption đã chọn. Ngưỡng >1.8: timelapse, <0.6: slow-motion; nhãn được chèn vào prefix caption trước khi lưu. Không xuất file score/label. Predictor dùng nguyên số frame checkpoint (16), nhịp tham chiếu 7.5 FPS, 3 lượt thích nghi theo code gốc, không nội suy vị trí. Shot ngắn hơn cửa sổ gốc báo unknown và giữ prefix thường. Log trực tiếp gồm OMNI decode/window/shot, SPEED từng lượt và nhãn cuối, CAPTION-FINAL từng shot, JSON-CAPTION sau gộp. Không ghi file log.

SpeedPredictor dùng NVDEC SimpleDecoder đồng bộ (`decoder: simple`) để tránh constructor ThreadedDecoder đã segfault trên Kaggle. Vẫn trả RGBP trong GPU memory; không CPU fallback. Nếu thấy exit -11/SIGSEGV, đó là lỗi native chứ không phải mất log. Cần chạy lại cell setup định nghĩa run() và cell config sau khi cập nhật notebook. SimpleDecoder vẫn cần kiểm tra thực tế với driver/codec của phiên Kaggle.

NVDEC speed inference được tách sang process mới `[SPEED-WORKER]`, không import Decord/PyAV/OpenCV/torchvision/OmniShotCut để cách ly thư viện native. Nhãn trao đổi qua stdin/stdout trong RAM; không tạo file score. Log worker vẫn hiển thị trực tiếp. Đây là biện pháp cách ly cần thử lại trên GPU; chưa xác nhận nguyên nhân sâu của SIGSEGV.

Thử InternVL3.5-2B hoặc VideoLLaMA3-2B ở cell 3. InternVL dùng tile theo config model (448), không dùng short_side của VideoLLaMA3. Frame mẫu vẫn đều theo frame_sample_fps. Để so sánh cùng video trên Drive, dùng folder đích riêng cho mỗi model ở cell 1 rồi chạy lại cell 4; mặc định video đã có JSON trên Drive sẽ bị bỏ qua.


In [ ]:
# 1. Cấu hình
import importlib.metadata, importlib.util, json, os, re, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path
REPO_URL = 'https://github.com/iamtro-2006/event-retrieval-system.git'
REPO_BRANCH = 'vbs-dev'  # nhánh chứa pipeline caption
DATASET = 'V3C'
START_VIDEO_ID = 1
END_VIDEO_ID = 10
VIDEO_EXTENSION = '.mp4'
SOURCE_ROOT_FOLDER_ID = '19b7skL2b0PRJWWweWFh8qVNTLQDF6x9K'
DEST_ROOT_FOLDER_ID = '1I97-3-Y_V0-dfk-tPS4g7H8lpAHyeu2r'
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf')
RCLONE_REMOTE = 'iamtro'
UPLOAD_BATCH_SIZE = 10
OVERWRITE_REMOTE = False

def video_id(number):
    return f'{number:05d}'

if DATASET != 'V3C' or not 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 99999:
    raise ValueError('Cần 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 99999 cho V3C')
if SOURCE_ROOT_FOLDER_ID == DEST_ROOT_FOLDER_ID:
    raise ValueError('Folder Drive nguồn và đích phải khác nhau')
if not RCLONE_CONFIG_SOURCE.is_file():
    raise FileNotFoundError(f'Chưa gắn rclone config: {RCLONE_CONFIG_SOURCE}')
WORK = Path('/kaggle/working/caption_pipeline')
WORK.mkdir(parents=True, exist_ok=True)
JOBS_ROOT = WORK / 'jobs'
JOBS_ROOT.mkdir(exist_ok=True)
RCLONE_CONFIG = WORK / 'rclone.conf'
shutil.copyfile(RCLONE_CONFIG_SOURCE, RCLONE_CONFIG)
RCLONE_CONFIG.chmod(0o600)
REPO_DIR = WORK / 'event-retrieval-system'
BACKEND_DIR = REPO_DIR / 'backend'
print('Workspace:', WORK)


In [ ]:
# 2. Clone/pull repo, cài dependencies chung + OmniShotCut và chuẩn bị rclone
def run(args, *, cwd=None, env=None):
    from collections import deque
    import queue
    import signal
    import threading
    import time
    args = [str(item) for item in args]
    print('+', ' '.join(args[:5]), '...' if len(args) > 5 else '', flush=True)
    tail = deque(maxlen=80)
    messages = queue.Queue()
    started = time.monotonic()
    last_output = started
    with subprocess.Popen(args, cwd=cwd, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, errors='replace',
                          bufsize=1) as process:
        def pump():
            try:
                for line in process.stdout:
                    messages.put(line)
            finally:
                messages.put(None)
        threading.Thread(target=pump, daemon=True).start()
        eof = False
        try:
            while not eof or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                except queue.Empty:
                    if time.monotonic() - last_output >= 30:
                        status = process.poll()
                        print(f'[RUN-HEARTBEAT] pid={process.pid} '
                              f'elapsed={time.monotonic()-started:.0f}s '
                              f'status={"running" if status is None else status}; '
                              'no new child output for 30s (not a progress estimate)', flush=True)
                        last_output = time.monotonic()
                    continue
                if line is None:
                    eof = True
                    continue
                print(line, end='', flush=True)
                tail.append(line)
                last_output = time.monotonic()
            exit_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            print('[RUN-END] interrupted by user', flush=True)
            raise
    detail = ''
    if exit_code < 0:
        try:
            detail = f' signal={signal.Signals(-exit_code).name}'
        except ValueError:
            detail = f' signal={-exit_code}'
    print(f'[RUN-END] pid={process.pid} exit_code={exit_code}{detail} '
          f'elapsed={time.monotonic()-started:.1f}s', flush=True)
    if exit_code:
        raise RuntimeError(f'Command exited with code {exit_code}{detail}: {args[:5]}\n'
                           + ''.join(tail))
    return exit_code

if not REPO_DIR.exists():
    run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    branch = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
    if branch != REPO_BRANCH:
        raise RuntimeError(f'Checkout đang ở {branch}, cần {REPO_BRANCH}')
    run(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if not (BACKEND_DIR / 'src/caption_extraction/pipeline/extract_caption.py').is_file():
    raise RuntimeError('Nhánh GitHub chưa có caption extraction; hãy push code lên vbs-dev')
print('Caption commit:', commit)

# Cài tối thiểu, giữ PyTorch GPU có sẵn trên Kaggle; không tạo venv.
# Transformers được cài theo model ở cell 3, không import trong kernel notebook.
if any(importlib.util.find_spec(name) is None
       for name in ('decord', 'ffmpeg', 'yaml', 'cv2', 'PyNvVideoCodec', 'safetensors')):
    run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed',
         'decord', 'ffmpeg-python', 'PyYAML', 'opencv-python-headless',
         'PyNvVideoCodec>=2.0.2,<3', 'safetensors'])
if importlib.util.find_spec('omnishotcut') is None:
    run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed',
         'git+https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git'])
import yaml
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Cần bật GPU trong Kaggle Settings')
print('GPU:', torch.cuda.get_device_name(0))

rclone = shutil.which('rclone')
if rclone is None and shutil.which('apt-get'):
    try:
        run(['apt-get', 'update', '-qq'])
        run(['apt-get', 'install', '-y', '-qq', 'rclone'])
        rclone = shutil.which('rclone')
    except subprocess.CalledProcessError:
        pass
if rclone is None:
    release = 'v1.75.2'
    archive = WORK / f'rclone-{release}-linux-amd64.zip'
    url = f'https://github.com/rclone/rclone/releases/download/{release}/{archive.name}'
    with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'kaggle-caption'}), timeout=120) as src, archive.open('wb') as dst:
        shutil.copyfileobj(src, dst)
    with zipfile.ZipFile(archive) as zf:
        matches = [item for item in zf.infolist() if item.filename.endswith('/rclone') and not item.is_dir()]
        if len(matches) != 1:
            raise RuntimeError('Không tìm thấy binary rclone trong ZIP release')
        rclone = WORK / 'rclone'
        with zf.open(matches[0]) as src, rclone.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    rclone.chmod(0o755)
run([rclone, 'version'])


In [ ]:
# 3. Thử nghiệm caption: sửa model, FPS, profile và prompt rồi chạy lại cell này + cell 5.
# Không cần sửa/push YAML mỗi lần. Thứ tự profile là thứ tự fallback khi OOM.
import hashlib
import math
import yaml

CAPTION_MODEL = 'OpenGVLab/InternVL3_5-2B'
# CAPTION_MODEL = 'DAMO-NLP-SG/VideoLLaMA3-2B'
INTERNVL_MAX_TILES = 1  # Default video: 1 tile/frame; >1 adds spatial tiles + thumbnail
FRAME_SAMPLE_FPS = 1.5
TIMELAPSE_THRESHOLD = 1.8      # score > ngưỡng này
SLOW_MOTION_THRESHOLD = 0.6    # score < ngưỡng này
VIDEOLLAMA_PROFILES = [
    {'max_frames': 16, 'short_side': 256},
    {'max_frames': 16, 'short_side': 224},
    {'max_frames': 12, 'short_side': 224},
    {'max_frames': 8, 'short_side': 224},
]

# InternVL keeps its trained tile size; reduce frame count on CUDA OOM.
INTERNVL_PROFILES = [
    {'max_frames': 16, 'short_side': 448},
    {'max_frames': 12, 'short_side': 448},
    {'max_frames': 8, 'short_side': 448},
]
IS_INTERNVL = 'internvl' in CAPTION_MODEL.lower()
CAPTION_PROFILES = INTERNVL_PROFILES if IS_INTERNVL else VIDEOLLAMA_PROFILES
if type(INTERNVL_MAX_TILES) is not int or INTERNVL_MAX_TILES < 1:
    raise ValueError('INTERNVL_MAX_TILES must be a positive integer')
if IS_INTERNVL and CAPTION_MODEL.lower().endswith('-hf'):
    raise ValueError('Select OpenGVLab/InternVL3_5-2B without -HF')

# Each CLI run uses a fresh process, so switching Transformers needs no kernel restart.
def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

requirements = {'transformers': '4.52.4', 'accelerate': '1.0.1'} if IS_INTERNVL else {
    'transformers': '4.46.3', 'accelerate': '1.0.1'}
extra = ['timm', 'einops', 'sentencepiece'] if IS_INTERNVL else []
if (any(installed_version(name) != version for name, version in requirements.items())
        or any(installed_version(name) is None for name in extra)):
    run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed',
         *[f'{name}=={version}' for name, version in requirements.items()], *extra])
print('Caption dependencies:', requirements, flush=True)

CAPTION_PROMPT = (
    'Describe the following video segment in English. Begin with "The video segment". In 1-4 detailed sentences, focusing main visible subjects, their distinctive details and positions, the relevant surroundings at that moment. Describe what happens from the beginning to the end in reasonable chronological order, including meaningful changes, interactions between subjects, and camera movement when visible. Be specific about observable actions, scene, interaction and subject attribute details and identity rather than giving a broad summary. Include only what can be seen; omit uncertain or absent details without mentioning their absence. DO NOT GUESS any event, action or intent.'
)

# Kiểm tra trước khi ghi cấu hình cho subprocess.
if not math.isfinite(FRAME_SAMPLE_FPS) or FRAME_SAMPLE_FPS <= 0:
    raise ValueError('FRAME_SAMPLE_FPS phải là số dương hữu hạn')
if not (math.isfinite(SLOW_MOTION_THRESHOLD) and math.isfinite(TIMELAPSE_THRESHOLD)
        and 0 < SLOW_MOTION_THRESHOLD < TIMELAPSE_THRESHOLD):
    raise ValueError('Cần 0 < SLOW_MOTION_THRESHOLD < TIMELAPSE_THRESHOLD, các ngưỡng hữu hạn')
if not CAPTION_PROMPT.strip() or not CAPTION_MODEL.strip():
    raise ValueError('Model và prompt không được rỗng')
if not CAPTION_PROFILES or any(
    not isinstance(profile, dict) or
    any(type(profile.get(key)) is not int or profile[key] <= 0
        for key in ('max_frames', 'short_side'))
    for profile in CAPTION_PROFILES
):
    raise ValueError('Mỗi profile cần max_frames và short_side là số nguyên dương')

config_path = BACKEND_DIR / 'configs/caption_extraction.yaml' 
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
cfg['extraction']['speed_predictor']['timelapse_threshold'] = float(TIMELAPSE_THRESHOLD)
cfg['extraction']['speed_predictor']['slow_motion_threshold'] = float(SLOW_MOTION_THRESHOLD)
cfg['extraction']['caption_model'] = CAPTION_MODEL
cfg['extraction']['internvl'] = {'max_tiles': INTERNVL_MAX_TILES}
cfg['extraction']['prompt'] = CAPTION_PROMPT.strip()
cfg['extraction']['frame_sample_fps'] = float(FRAME_SAMPLE_FPS)
cfg['extraction']['caption_profiles'] = [dict(profile) for profile in CAPTION_PROFILES]
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
CAPTION_RUN_ID = hashlib.sha256(yaml.safe_dump(cfg['extraction'], sort_keys=True).encode()).hexdigest()[:10]
STAGE_DIR = WORK / 'pending_json_2b' / CAPTION_RUN_ID
STAGE_DIR.mkdir(parents=True, exist_ok=True)
print('Model:', CAPTION_MODEL, '| prompt run:', CAPTION_RUN_ID)

print('SpeedPredictor:', cfg['extraction']['speed_predictor'], flush=True)
print('Live logs: [OMNI] -> [OMNI-SHOT] -> [SPEED] -> [CAPTION-FINAL] -> [JSON-CAPTION]', flush=True)

print('Caption sample FPS:', FRAME_SAMPLE_FPS, flush=True)
print('Caption fallback profiles:', CAPTION_PROFILES, flush=True)
if IS_INTERNVL:
    print('InternVL uses model tile size (448), ignores short_side; max_tiles:', INTERNVL_MAX_TILES, flush=True)
print('Caption prompt:', CAPTION_PROMPT, flush=True)
print('Local captions:', WORK / 'captions_2b' / CAPTION_RUN_ID / DATASET, flush=True)
print('Đã áp dụng. Chạy lại cell 5 để xử lý; việc bỏ qua video đã có trên Drive vẫn theo cấu hình notebook.', flush=True)


In [ ]:
# 4. Liệt kê video nguồn và bỏ qua caption JSON đã upload
def remote_args(folder_id):
    return [rclone, '--config', RCLONE_CONFIG, '--drive-root-folder-id', folder_id]

source_names = set(subprocess.check_output(
    [*map(str, remote_args(SOURCE_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
selected = [video_id(number) for number in range(START_VIDEO_ID, END_VIDEO_ID + 1)]
missing = [vid for vid in selected if f'{vid}{VIDEO_EXTENSION}' not in source_names]
if missing:
    raise FileNotFoundError(f'Thiếu video trong Drive nguồn: {missing}')
dest_names = set(subprocess.check_output(
    [*map(str, remote_args(DEST_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
pending = [vid for vid in selected if OVERWRITE_REMOTE or f'{vid}.json' not in dest_names]
print(f'Đã chọn {len(selected)} video; cần xử lý {len(pending)}:', pending)


In [ ]:
# 5. Từng lô: tải video cần xử lý, tạo shot + caption, upload JSON rồi dọn video tạm
import math
cfg = yaml.safe_load((BACKEND_DIR / 'configs/caption_extraction.yaml').read_text(encoding='utf-8'))
if cfg['extraction']['caption_model'] != CAPTION_MODEL:
    raise RuntimeError('Caption config trên nhánh này chưa đúng model')
CAPTION_ENV = os.environ.copy()
CAPTION_ENV['PYTHONPATH'] = str(BACKEND_DIR)
CAPTION_ENV['PYTHONUNBUFFERED'] = '1'
CAPTION_ENV['PYTHONFAULTHANDLER'] = '1'
CAPTION_ENV['HF_HOME'] = str(WORK / 'hf_cache')
CAPTION_ENV['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
SCENES_DIR = WORK / 'scenes' / DATASET
CAPTIONS_DIR = WORK / 'captions_2b' / CAPTION_RUN_ID / DATASET
SCENES_DIR.mkdir(parents=True, exist_ok=True)
CAPTIONS_DIR.mkdir(parents=True, exist_ok=True)

def validate_json(path, vid):
    data = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(data, list) or not data:
        raise ValueError(f'{path}: cần danh sách caption không rỗng')
    previous_start = -1.0
    for item in data:
        if (not isinstance(item, dict) or set(item) != {'video_id', 'start', 'end', 'caption'} or
                item['video_id'] != vid or not isinstance(item['caption'], str) or
                not item['caption'].strip() or
                not isinstance(item['start'], (int, float)) or
                not isinstance(item['end'], (int, float)) or
                not math.isfinite(item['start']) or not math.isfinite(item['end']) or
                not 0 <= item['start'] < item['end'] or item['start'] < previous_start):
            raise ValueError(f'{path}: segment không đúng schema')
        previous_start = item['start']
    return len(data)

# Restore staged JSON after an interrupted upload; completed files remain in CAPTIONS_DIR.
for staged in STAGE_DIR.glob('*.json'):
    local = CAPTIONS_DIR / staged.name
    if not local.is_file() or staged.read_bytes() != local.read_bytes():
        raise RuntimeError(f'JSON staged không khớp bản local: {staged}')
    staged.unlink()

# Recheck remote on cell rerun, so completed batches are not uploaded again.
remote_done = set(subprocess.check_output(
    [*map(str, remote_args(DEST_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
remaining = [vid for vid in selected if OVERWRITE_REMOTE or f'{vid}.json' not in remote_done]
completed = {}
for batch_number, start in enumerate(range(0, len(remaining), UPLOAD_BATCH_SIZE)):
    batch_ids = remaining[start:start + UPLOAD_BATCH_SIZE]
    batch_work = JOBS_ROOT / f'batch_{batch_number:04d}'
    video_dir = batch_work / 'videos' / DATASET
    video_dir.mkdir(parents=True, exist_ok=True)
    need_processing = [vid for vid in batch_ids if not (CAPTIONS_DIR / f'{vid}.json').is_file()]
    for vid in need_processing:
        path = video_dir / f'{vid}{VIDEO_EXTENSION}'
        if not path.is_file() or path.stat().st_size == 0:
            run([*remote_args(SOURCE_ROOT_FOLDER_ID), 'copyto',
                 f'{RCLONE_REMOTE}:{path.name}', path])
        else:
            print('Dùng lại video đã tải:', path.name)
        if not path.is_file() or path.stat().st_size == 0:
            raise RuntimeError(f'Video tải về rỗng: {path.name}')
    if need_processing:
        cmd = [sys.executable, '-u', '-m', 'scripts.caption.extract.run', '--dataset', DATASET.lower(),
               '--input-dir', video_dir, '--scenes-dir', SCENES_DIR, '--output-dir', CAPTIONS_DIR]
        for vid in need_processing:
            cmd += ['--video-id', vid]
        run(cmd, cwd=BACKEND_DIR, env=CAPTION_ENV)
    for vid in batch_ids:
        result = CAPTIONS_DIR / f'{vid}.json'
        if not result.is_file():
            raise FileNotFoundError(f'Không có caption JSON: {result}')
        completed[vid] = validate_json(result, vid)
        shutil.copy2(result, STAGE_DIR / result.name)
        print(f'Caption xong {vid}: {completed[vid]} segment')
    for path in video_dir.glob(f'*{VIDEO_EXTENSION}'):
        path.unlink()  # giữ shot/JSON trong WORK, giải phóng video trước khi upload
    run([*remote_args(DEST_ROOT_FOLDER_ID), 'copy', STAGE_DIR, f'{RCLONE_REMOTE}:',
         '--transfers', str(min(UPLOAD_BATCH_SIZE, len(batch_ids))), '--checkers', '10'])
    run([*remote_args(DEST_ROOT_FOLDER_ID), 'check', STAGE_DIR, f'{RCLONE_REMOTE}:', '--one-way'])
    for path in STAGE_DIR.glob('*.json'):
        path.unlink()
    shutil.rmtree(batch_work)
    print(f'Đã upload {len(batch_ids)} caption JSON; giữ shot/JSON local, dọn video tạm')
print('Hoàn tất:', completed, '| commit:', commit)
